# Lab 8 : Check the claims

In [1]:
import pandas as pd

listings = pd.read_csv("data/listings_clean.csv", parse_dates=["first_review", "last_review"])
print(listings.shape)

(4574, 73)


## Task 1 : Claim 1 "A night in Stockholm costs 1,791."

### Check the median and how many listings have a price

In [2]:
price = listings["price"]
print("median:", price.median())
print("with a price:", price.notna().sum())
print("without a price:", price.isna().sum(), round(price.isna().mean() * 100, 1), "%")

median: 1791.0
with a price: 2957
without a price: 1617 35.4 %


### What is common for the listings with no price?

In [3]:
no_price = listings[price.isna()]
print("no price and 0 open days:", (no_price["availability_365"] == 0).sum(), "of", len(no_price))

no price and 0 open days: 1490 of 1617


### Long stays (28 nights or more) compared to the rest

In [4]:
with_price = listings[price.notna()]
long_stay = with_price["minimum_nights"] >= 28

print("long stays:", long_stay.sum(), " median:", with_price[long_stay]["price"].median())
print("others:", (~long_stay).sum(), " median:", with_price[~long_stay]["price"].median())

long stays: 97  median: 708.89
others: 2860  median: 1839.25


**What I found:**
- 1791 is right, but it is the median, and only for the 2957 listings that have a price.
- 1617 listings (35 %) have no price. Most of them (1490) are not open any day, so you can't book them anyway.
- Listings with 28+ nights minimum are much cheaper (709) than the others (1839).
- The sentence doesn't say SEK, it doesn't say the date, and it doesn't say it's the median.

**Verdict:** Holds with reservations.

**My sentence:** "In September 2026 the median price for one night was 1,791 SEK, for the 2,957 Stockholm listings that had a price (65 % of all)." 

## Task 2 : Claim 2 "The listings with a perfect 5.00 rating are the best in Stockholm."

### How many listings have a rating, and how many have 5.00

In [5]:
rated = listings[listings["has_reviews"]]
perfect = rated["review_scores_rating"] == 5

print("rated listings:", len(rated))
print("perfect 5.00:", perfect.sum(), round(perfect.mean() * 100, 1), "%")

rated listings: 3930
perfect 5.00: 1484 37.8 %


### How many reviews do the 5.00 listings have?

In [6]:
perfect_rows = rated[perfect]
print("perfect with 20+ reviews:", (perfect_rows["number_of_reviews"] >= 20).sum())
print("share with 20+:", round((perfect_rows["number_of_reviews"] >= 20).mean() * 100, 1), "%")
print("median reviews, perfect:", perfect_rows["number_of_reviews"].median())
print("median reviews, others:", rated[~perfect]["number_of_reviews"].median())

perfect with 20+ reviews: 130
share with 20+: 8.8 %
median reviews, perfect: 4.0
median reviews, others: 25.0


**What I found:**
- 3930 listings have a rating. 1484 of them (38 %) have 5.00.
- Only 130 of those have 20 reviews or more. So most of them (91 %) have less than 20 reviews.
- A 5.00 listing has 4 reviews (median). The others have 25.
- It is easy to get 5.00 when only a few people reviewed. With many guests, someone will give 4.

**Verdict:** Does not hold.

**My sentence:** "38 % of the rated listings have 5.00, but 91 % of them have less than 20 reviews, so 5.00 doesn't show which ones are best." 

## Task 3 : Claim 3 "The market has grown: only 131 listings started in 2016, but 610 in 2024."

### Count listings per year of first review

In [7]:
listings["first_year"] = listings["first_review"].dt.year
listings["first_year"].value_counts().sort_index()

first_year
2011.0      2
2012.0     12
2013.0     26
2014.0     51
2015.0    116
2016.0    131
2017.0    148
2018.0    191
2019.0    234
2020.0     96
2021.0    159
2022.0    332
2023.0    565
2024.0    610
2025.0    542
2026.0    715
Name: count, dtype: int64

### How many of them got a review in 2026?

In [8]:
listings["reviewed_2026"] = listings["last_review"].dt.year == 2026
share = listings.groupby("first_year")["reviewed_2026"].mean().round(2)
print("2016:", share[2016])
print("2024:", share[2024])

2016: 0.53
2024: 0.67


**What I found:**
- The numbers are right: 131 in 2016 and 610 in 2024.
- But the table only has listings that still exist now. If a listing started in 2016 and closed later, it is not here. So 2016 looks smaller than it really was.
- Old listings are also less active: 53 % of the 2016 ones got a review in 2026, and 67 % of the 2024 ones.

**Verdict:** Cannot be concluded from this table.

**My sentence:** "Of the listings that still exist, 131 got their first review in 2016 and 610 in 2024. We can't say how many started in those years." 

## Task 4 : Claim 4 "Most hosts in Stockholm are professional companies with many listings."

### Count the hosts (one host can have many listings)

In [9]:
print("listings:", len(listings))
print("hosts:", listings["host_id"].nunique())

listings: 4574
hosts: 3591


### How many listings each host has

In [10]:
per_host = listings.groupby("host_id").size()

one = (per_host == 1).sum()
many = (per_host >= 2).sum()
print("hosts with 1 listing:", one, round(one / len(per_host) * 100, 1), "%")
print("hosts with 2+ listings:", many, round(many / len(per_host) * 100, 1), "%")
print("listings they own:", per_host[per_host >= 2].sum(), round(per_host[per_host >= 2].sum() / len(listings) * 100, 1), "%")
print("largest host:", per_host.max())

hosts with 1 listing: 3237 90.1 %
hosts with 2+ listings: 354 9.9 %
listings they own: 1337 29.2 %
largest host: 49


**What I found:**
- 4574 listings, but only 3591 hosts.
- 3237 hosts (90 %) have just one listing. So most hosts are normal people, not big companies.
- 354 hosts have two or more, and they have 1337 listings together (29 %). The biggest host has 49.
- There is no column that tells if a host is a company, so I can't check that part.

**Verdict:** Does not hold (and the "company" part can't be checked).

**My sentence:** "90 % of the hosts have only one listing, but the 10 % with more than one have 29 % of all listings." 

## Task 5 : Data note

### Missing values and estimated columns

In [11]:
print("price missing:", round(listings["price"].isna().mean() * 100, 1), "%")
print("rating missing:", round(listings["review_scores_rating"].isna().mean() * 100, 1), "%")
print([c for c in listings.columns if c.startswith("estimated_")])

price missing: 35.4 %
rating missing: 14.1 %
['estimated_occupancy_l365d', 'estimated_revenue_l365d']


### Data note
The table has 4574 Airbnb listings in Stockholm, taken on 26 and 27 September 2026.
It comes from Inside Airbnb (insideairbnb.com), licence CC BY 4.0, so we have to give them credit.
35 % of the prices are missing, mostly for listings that are not open any day.
14 % of the ratings are missing because those listings have no reviews yet.
Closed listings are not in the table, and hotels on other sites are not in it either.
estimated_occupancy_l365d and estimated_revenue_l365d are calculated, not real measured numbers.
The table can't tell us why a price is high or low, or if guests were really happy.
The claim I would fix first in a newspaper is claim 2, because people would pick a place with 3 reviews over one with 300 and think it's the best.